# lltCapstoneGPT2 exact-resume check (2x T4)

Runs the H100 reference script on a small budget: 12 steps straight, then a fresh process resumed from step 6, then a branch decay from step 6.

In [ ]:

import json, os, platform, subprocess, sys, time
T_START = time.time()
RUN = {"module": "lltCapstoneGPT2", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
       "probes": [], "benches": [], "log": {}}
os.makedirs("/kaggle/working/data", exist_ok=True)

def save():
    import torch
    RUN["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    for mod in ("transformers", "datasets", "tiktoken", "numpy", "lm_eval"):
        try:
            RUN["env"][mod] = __import__(mod).__version__
        except Exception:
            pass
    RUN["elapsedSec"] = round(time.time() - T_START, 1)
    RUN["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    with open("/kaggle/working/lltCapstoneGPT2_resume.json", "w") as f:
        json.dump(RUN, f, indent=1, default=str)

def sh(key, cmd, timeout=900, env=None):
    t = time.time()
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=env)
        out, rc = p.stdout, p.returncode
        if rc != 0:
            out += "\nSTDERR TAIL\n" + p.stderr[-3000:]
    except subprocess.TimeoutExpired as e:
        out, rc = "TIMEOUT " + str(e)[-500:], -9
    RUN["log"][key] = {"rc": rc, "sec": round(time.time() - t, 1), "out": out[-6000:]}
    print(key, "rc", rc, round(time.time() - t, 1), "s")
    print(out[-3000:])
    save()
    return out

def result_line(out):
    for line in out.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[7:])
        if line.startswith("{"):
            try:
                return json.loads(line)
            except Exception:
                pass
    return None

import torch
print(platform.python_version(), torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
try:
    import tiktoken
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
save()


In [ ]:
with open('gpt.py', 'w') as f:
    f.write('# gpt.py - the course GPT (lltGPT: RMSNorm, RoPE, SwiGLU, no biases, tied embeddings) at capstone sizes.\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nCONFIGS = {\n    "124m": dict(V=50257, d=768, L=12, H=12, ctx=1024),   # GPT-2 small shape\n    "40m": dict(V=50257, d=384, L=12, H=6, ctx=1024),     # same depth, half the width\n}\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps=1e-5):\n        super().__init__()\n        self.w = nn.Parameter(torch.ones(d))\n        self.eps = eps\n\n    def forward(self, x):\n        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)\n\n\ndef rope_tables(T, hd, device):\n    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))\n    f = torch.outer(torch.arange(T, device=device).float(), inv)\n    return f.cos(), f.sin()\n\n\ndef apply_rope(x, cos, sin):\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)\n    return out.flatten(-2).type_as(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h):\n        super().__init__()\n        self.h = h\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.qkv = nn.Linear(d, 3 * d, bias=False)\n        self.o = nn.Linear(d, d, bias=False)\n        hid = int(8 * d / 3 / 64 + 0.999) * 64\n        self.w1 = nn.Linear(d, hid, bias=False)\n        self.w3 = nn.Linear(d, hid, bias=False)\n        self.w2 = nn.Linear(hid, d, bias=False)\n\n    def forward(self, x, cos, sin):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))\n        h = self.n2(x)\n        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))\n\n\nclass GPT(nn.Module):\n    def __init__(self, V, d, L, H, ctx, ckpt=False):\n        super().__init__()\n        self.emb = nn.Embedding(V, d)\n        self.blocks = nn.ModuleList([Block(d, H) for _ in range(L)])\n        self.norm = RMSNorm(d)\n        self.head = nn.Linear(d, V, bias=False)\n        self.head.weight = self.emb.weight            # tied, as in GPT-2\n        self.H, self.ctx, self.ckpt = H, ctx, ckpt\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                std = 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02\n                nn.init.normal_(p, 0.0, std)\n\n    def forward(self, idx, targets=None):\n        T = idx.shape[1]\n        cos, sin = rope_tables(T, self.emb.embedding_dim // self.H, idx.device)\n        x = self.emb(idx)\n        for b in self.blocks:\n            x = checkpoint(b, x, cos, sin, use_reentrant=False) if self.ckpt else b(x, cos, sin)\n        logits = self.head(self.norm(x))\n        if targets is None:\n            return logits                                  # used by generation and the Hugging Face wrapper\n        return F.cross_entropy(logits.float().flatten(0, 1), targets.flatten())\n\n\ndef flops_per_token(cfg, n_params):\n    # nanoGPT / PaLM convention: 6N for the matmuls plus 12*L*d*T for attention scores and values\n    return 6 * n_params + 12 * cfg["L"] * cfg["d"] * cfg["ctx"]\n')
print('wrote', 'gpt.py')


In [ ]:
with open('prep_fineweb.py', 'w') as f:
    f.write('# prep_fineweb.py - FineWeb-Edu sample-10BT -> GPT-2 tokens, one uint16 shard per parquet file.\n#\n#   python prep_fineweb.py --out /workspace/data                     # all 14 files, about 10B tokens\n#   python prep_fineweb.py --out /workspace/data --files 1           # calibration: one file\n#   python prep_fineweb.py --out /workspace/hq --min-int-score 4     # the ablation\'s high-score subset\n#\n# Layout: <out>/val.bin (the first --val-tokens tokens of file 000), <out>/train_XXX.bin, <out>/index.json.\n# Every document starts with the end-of-text token 50256, as in llm.c. Tokens are stored as uint16 (vocab 50,257).\nimport argparse, json, os, time\nfrom multiprocessing import Pool\n\np = argparse.ArgumentParser()\np.add_argument("--out", required=True)\np.add_argument("--files", type=int, default=14)            # sample/10BT has 14 parquet files (checked 2026-11-05)\np.add_argument("--val-tokens", type=int, default=10_485_760)\np.add_argument("--max-docs", type=int, default=0)          # per file; 0 = all (tests use a small number)\np.add_argument("--min-int-score", type=int, default=0)     # FineWeb-Edu classifier score filter; 0 = keep all\np.add_argument("--workers", type=int, default=0)\na = p.parse_args()\n\n\ndef one_file(i):\n    import numpy as np\n    import pyarrow.parquet as pq\n    import tiktoken\n    from huggingface_hub import hf_hub_download\n    t0 = time.time()\n    path = hf_hub_download("HuggingFaceFW/fineweb-edu", f"sample/10BT/{i:03d}_00000.parquet", repo_type="dataset")\n    t_dl = time.time() - t0\n    enc = tiktoken.get_encoding("gpt2")\n    eot = enc.eot_token\n    out = os.path.join(a.out, f"raw_{i:03d}.bin")\n    n_tok = n_doc = n_chr = 0\n    with open(out, "wb") as f:\n        cols = ["text", "int_score"] if a.min_int_score else ["text"]\n        for batch in pq.ParquetFile(path).iter_batches(batch_size=2048, columns=cols):\n            texts = batch.column(0).to_pylist()\n            if a.min_int_score:\n                texts = [t for t, s in zip(texts, batch.column(1).to_pylist()) if s >= a.min_int_score]\n            if a.max_docs:\n                texts = texts[: max(0, a.max_docs - n_doc)]\n            if not texts:\n                if a.max_docs and n_doc >= a.max_docs:\n                    break\n                continue\n            ids = enc.encode_ordinary_batch(texts, num_threads=2)\n            arr = np.concatenate([np.array([eot] + x, dtype=np.uint16) for x in ids])\n            f.write(arr.tobytes())\n            n_tok += len(arr)\n            n_doc += len(texts)\n            n_chr += sum(len(t) for t in texts)\n            if a.max_docs and n_doc >= a.max_docs:\n                break\n    return {"file": i, "tokens": n_tok, "docs": n_doc, "chars": n_chr, "downloadSec": round(t_dl, 1),\n            "sec": round(time.time() - t0, 1)}\n\n\nif __name__ == "__main__":\n    os.makedirs(a.out, exist_ok=True)\n    t0 = time.time()\n    workers = a.workers or max(1, min(a.files, (os.cpu_count() or 2) // 2))\n    with Pool(workers) as pool:\n        parts = pool.map(one_file, range(a.files))\n    import numpy as np\n    # Split file 000 into the fixed validation slice and its training remainder, then name the train shards.\n    raw0 = np.memmap(os.path.join(a.out, "raw_000.bin"), dtype=np.uint16, mode="r")\n    if len(raw0) <= a.val_tokens:\n        raise SystemExit(f"file 000 has {len(raw0)} tokens, fewer than --val-tokens {a.val_tokens}")\n    raw0[: a.val_tokens].tofile(os.path.join(a.out, "val.bin"))\n    shards = []\n    for part in parts:\n        i = part["file"]\n        src = os.path.join(a.out, f"raw_{i:03d}.bin")\n        dst = os.path.join(a.out, f"train_{i:03d}.bin")\n        start = a.val_tokens if i == 0 else 0\n        os.replace(src, dst)\n        shards.append({"file": os.path.basename(dst), "tokens": part["tokens"], "start": start})\n    del raw0\n    total = sum(s["tokens"] - s["start"] for s in shards)\n    import tiktoken\n    index = {"dataset": "HuggingFaceFW/fineweb-edu", "config": "sample-10BT", "licence": "ODC-By 1.0",\n             "tokenizer": "gpt2 (tiktoken " + tiktoken.__version__ + ")", "eot": 50256, "dtype": "uint16",\n             "minIntScore": a.min_int_score, "maxDocsPerFile": a.max_docs,\n             "val": {"file": "val.bin", "tokens": a.val_tokens}, "train": shards, "trainTokens": total,\n             "docs": sum(q["docs"] for q in parts), "chars": sum(q["chars"] for q in parts),\n             "perFile": parts, "workers": workers, "sec": round(time.time() - t0, 1)}\n    index["charsPerToken"] = round(index["chars"] / max(1, total + a.val_tokens), 3)\n    with open(os.path.join(a.out, "index.json"), "w") as f:\n        json.dump(index, f, indent=1)\n    print(json.dumps({k: v for k, v in index.items() if k not in ("perFile", "train")}))\n')
print('wrote', 'prep_fineweb.py')


In [ ]:
with open('train.py', 'w') as f:
    f.write('# train.py - pretrain the course GPT on FineWeb-Edu tokens, with a WSD schedule, exact resume and Run Library logs.\n#\n# One script for both paths:\n#   H100 reference run (1 GPU, bf16):   torchrun --nproc_per_node=1 train.py --data /workspace/data --out /workspace/run\n#   Kaggle 2x T4 (fp16 + GradScaler):   torchrun --nproc_per_node=2 train.py --micro 8 ...\n#   Resume after a stop:                 add --resume auto\n#   Learner decay from our checkpoint:   --resume <ckpt dir> --stable-end 4529 --max-steps 4769 --out <new dir>\n#\n# Schedule (WSD): linear warmup to --lr over --warmup steps, constant until --stable-end, then linear to 0 at --max-steps.\n# Defaults reproduce the reference run: 19,073 steps x 524,288 tokens = 10.0B tokens, decay over the last 10%.\nimport argparse, contextlib, datetime, glob, json, math, os, random, shutil, time\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom gpt import GPT, CONFIGS, flops_per_token\n\np = argparse.ArgumentParser()\np.add_argument("--data", required=True)\np.add_argument("--out", required=True)\np.add_argument("--run-id", default="lltCapstoneGPT2-ref-124m-10B")\np.add_argument("--size", default="124m", choices=sorted(CONFIGS))\np.add_argument("--micro", type=int, default=32)\np.add_argument("--tokens-per-step", type=int, default=524_288)\np.add_argument("--max-steps", type=int, default=19_073)\np.add_argument("--warmup", type=int, default=700)\np.add_argument("--stable-end", type=int, default=17_166)\np.add_argument("--lr", type=float, default=6e-4)\np.add_argument("--wd", type=float, default=0.1)\np.add_argument("--clip", type=float, default=1.0)\np.add_argument("--no-compile", action="store_true")\np.add_argument("--dtype", default="auto", choices=["auto", "bf16", "fp16"])\np.add_argument("--save-every", type=int, default=500)\np.add_argument("--keep-last", type=int, default=2)\np.add_argument("--milestones", default="954,1908,4529,4769,9537,17166,19073")\np.add_argument("--export-every", type=int, default=1000)\np.add_argument("--val-every", type=int, default=250)\np.add_argument("--val-tokens", type=int, default=4_194_304)     # first 4M tokens of val.bin\np.add_argument("--sample-every", type=int, default=1000)\np.add_argument("--resume", default="")\np.add_argument("--reset-loader-shard", type=int, default=-1)   # branch runs: start reading at this shard (seed = shard)\np.add_argument("--stop-after-sec", type=float, default=0)       # stop cleanly with a checkpoint (12 h sessions)\np.add_argument("--seed", type=int, default=0)\na = p.parse_args()\n\n# ---- process group, device, precision ------------------------------------------------------------------------\ndist.init_process_group("nccl", timeout=datetime.timedelta(minutes=30))\nrank, world = dist.get_rank(), dist.get_world_size()\ntorch.cuda.set_device(rank)\ndev = torch.device("cuda", rank)\ncap = torch.cuda.get_device_capability(rank)\n# Pick by capability, not torch.cuda.is_bf16_supported(): that returns True on a T4, which has no bf16 tensor cores.\ndtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(a.dtype, torch.bfloat16 if cap >= (8, 0) else torch.float16)\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\nmaster = rank == 0\ncfg = CONFIGS[a.size]\nT = cfg["ctx"]\naccum = a.tokens_per_step // (a.micro * T * world)\nassert accum * a.micro * T * world == a.tokens_per_step, "tokens-per-step must equal micro x 1024 x GPUs x accum"\nmilestones = {int(s) for s in a.milestones.split(",") if s}\nckpt_dir, w_dir = os.path.join(a.out, "ckpt"), os.path.join(a.out, "weights")\nif master:\n    os.makedirs(ckpt_dir, exist_ok=True)\n    os.makedirs(w_dir, exist_ok=True)\n\n\n# ---- data ----------------------------------------------------------------------------------------------------\nclass Loader:\n    """Reads contiguous windows across shards. Each micro-step takes world*b*T tokens; rank r reads slice r.\n    The state (shard, pos, epoch) is saved in every checkpoint, so a resumed run reads exactly the next tokens."""\n\n    def __init__(self, data_dir, b):\n        self.index = json.load(open(os.path.join(data_dir, "index.json")))\n        self.shards = [np.memmap(os.path.join(data_dir, s["file"]), dtype=np.uint16, mode="r") for s in self.index["train"]]\n        self.starts = [s["start"] for s in self.index["train"]]\n        self.b, self.shard, self.pos, self.epoch = b, 0, self.starts[0], 0\n\n    def state_dict(self):\n        return {"shard": self.shard, "pos": self.pos, "epoch": self.epoch}\n\n    def load_state_dict(self, s):\n        self.shard, self.pos, self.epoch = s["shard"], s["pos"], s["epoch"]\n\n    def next(self):\n        n = self.b * T\n        while self.pos + world * n + 1 > len(self.shards[self.shard]):\n            self.shard += 1\n            if self.shard == len(self.shards):\n                self.shard, self.epoch = 0, self.epoch + 1\n            self.pos = self.starts[self.shard]\n        s = self.pos + rank * n\n        w = torch.from_numpy(self.shards[self.shard][s:s + n + 1].astype(np.int64))\n        self.pos += world * n\n        x, y = w[:-1].view(self.b, T), w[1:].view(self.b, T)\n        return x.pin_memory().to(dev, non_blocking=True), y.pin_memory().to(dev, non_blocking=True)\n\n\nloader = Loader(a.data, a.micro)\nval_data = np.memmap(os.path.join(a.data, loader.index["val"]["file"]), dtype=np.uint16, mode="r")\n\n# ---- model and optimizer -------------------------------------------------------------------------------------\ntorch.manual_seed(a.seed)\nraw = GPT(**cfg).to(dev)\nN = sum(q.numel() for q in raw.parameters())\ndecay = [q for q in raw.parameters() if q.dim() >= 2]       # matrices and the tied embedding get weight decay\nno_decay = [q for q in raw.parameters() if q.dim() < 2]     # RMSNorm gains do not\nopt = torch.optim.AdamW([{"params": decay, "weight_decay": a.wd}, {"params": no_decay, "weight_decay": 0.0}],\n                        lr=a.lr, betas=(0.9, 0.95), eps=1e-8, fused=True)\nscaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)   # bf16 needs no loss scaling\nmodel = DDP(raw if a.no_compile else torch.compile(raw), device_ids=[rank])\n\n\ndef lr_at(step):                         # step counts completed optimizer steps; this is the LR for step+1\n    if step < a.warmup:\n        return a.lr * (step + 1) / a.warmup\n    if step < a.stable_end:\n        return a.lr\n    return a.lr * max(0.0, (a.max_steps - step) / max(1, a.max_steps - a.stable_end))\n\n\n# ---- checkpoints -----------------------------------------------------------------------------------------------\ndef save_ckpt(step, run):\n    final = os.path.join(ckpt_dir, f"step_{step:06d}")\n    tmp = final + ".tmp"\n    if master:\n        shutil.rmtree(tmp, ignore_errors=True)\n        os.makedirs(tmp)\n    dist.barrier()\n    torch.save({"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n                "cuda": torch.cuda.get_rng_state(dev)}, os.path.join(tmp, f"rng_{rank}.pt"))\n    if master:\n        torch.save({"model": raw.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),\n                    "loader": loader.state_dict(), "step": step, "args": vars(a), "config": cfg, "world": world,\n                    "micro": a.micro, "dtype": str(dtype), "metrics": run["metrics"], "samples": run["samples"]},\n                   os.path.join(tmp, "state.pt"))\n    dist.barrier()\n    if master:\n        shutil.rmtree(final, ignore_errors=True)\n        os.replace(tmp, final)                # a session that dies mid-save leaves the previous checkpoint intact\n        done = sorted(d for d in glob.glob(os.path.join(ckpt_dir, "step_*")) if not d.endswith(".tmp"))\n        rolling = [d for d in done if int(d[-6:]) not in milestones]\n        for d in rolling[:-a.keep_last]:\n            shutil.rmtree(d, ignore_errors=True)\n    dist.barrier()\n\n\ndef export_weights(step):\n    if master:\n        sd = {k: v.detach().to(torch.bfloat16).cpu() for k, v in raw.state_dict().items()}\n        torch.save({"config": cfg, "step": step, "tokens": step * a.tokens_per_step, "state_dict": sd},\n                   os.path.join(w_dir, f"weights_step_{step:06d}.pt"))\n\n\ndef find_resume():\n    if a.resume != "auto":\n        return a.resume\n    done = sorted(d for d in glob.glob(os.path.join(ckpt_dir, "step_*")) if not d.endswith(".tmp"))\n    return done[-1] if done else ""\n\n\n# ---- eval and samples --------------------------------------------------------------------------------------------\n@torch.no_grad()\ndef val_loss():\n    raw.eval()\n    n = a.micro * T\n    chunks = (min(a.val_tokens, len(val_data) - 1)) // n\n    tot, cnt = torch.zeros((), device=dev), 0\n    for c in range(rank, chunks, world):\n        w = torch.from_numpy(val_data[c * n:c * n + n + 1].astype(np.int64)).to(dev)\n        with torch.autocast("cuda", dtype=dtype):\n            tot += raw(w[:-1].view(a.micro, T), w[1:].view(a.micro, T))\n        cnt += 1\n    stats = torch.tensor([tot.item(), cnt], device=dev, dtype=torch.float64)\n    dist.all_reduce(stats)\n    raw.train()\n    return (stats[0] / stats[1]).item(), int(stats[1].item()) * n\n\n\nPROMPTS = ["The water cycle is", "In mathematics, a prime number", "The French Revolution began"]\n\n\n@torch.no_grad()\ndef samples(step):\n    import tiktoken\n    enc = tiktoken.get_encoding("gpt2")\n    raw.eval()\n    out = []\n    g = torch.Generator(device=dev).manual_seed(1234)\n    for prompt in PROMPTS:\n        ids = torch.tensor([enc.encode_ordinary(prompt)], device=dev)\n        for _ in range(48):\n            with torch.autocast("cuda", dtype=dtype):\n                logits = raw(ids)[:, -1, :].float()\n            v, ix = torch.topk(logits, 40)\n            nxt = ix.gather(1, torch.multinomial(torch.softmax(v / 0.8, -1), 1, generator=g))\n            ids = torch.cat([ids, nxt], 1)\n        out.append({"step": step, "prompt": prompt, "completion": enc.decode(ids[0].tolist())[len(prompt):]})\n    raw.train()\n    return out\n\n\ndef write_run(run, step, t_train):\n    if not master:\n        return\n    run.update({"wallClockSec": round(t_train, 1), "lastStep": step, "date": time.strftime("%Y-%m-%d")})\n    with open(os.path.join(a.out, a.run_id + ".json.tmp"), "w") as f:\n        json.dump(run, f, indent=1)\n    os.replace(os.path.join(a.out, a.run_id + ".json.tmp"), os.path.join(a.out, a.run_id + ".json"))\n\n\n# ---- resume ------------------------------------------------------------------------------------------------------\nstep, prior_sec = 0, 0.0\nrun = {"runId": a.run_id, "module": "lltCapstoneGPT2", "tier": "T4" if cap >= (8, 0) else "T2", "label": "baseline",\n       "hardware": {"gpu": torch.cuda.get_device_name(rank), "count": world, "capability": list(cap),\n                    "vramGB": round(torch.cuda.get_device_properties(rank).total_memory / 1e9, 1)},\n       "env": {"torch": torch.__version__, "cuda": torch.version.cuda, "dtype": str(dtype), "compile": not a.no_compile},\n       "notebookCommit": None, "config": {**vars(a), "params": N, "accum": accum, "model": cfg}, "seed": a.seed,\n       "metrics": [], "evals": {}, "samples": [], "resumes": [], "costUSD": None, "priceCheckedOn": None}\nsrc = find_resume()\nif src:\n    st = torch.load(os.path.join(src, "state.pt"), map_location="cpu", weights_only=False)\n    raw.load_state_dict(st["model"])\n    opt.load_state_dict(st["optimizer"])\n    for g in opt.param_groups:            # the schedule comes from this run\'s arguments, not the saved ones\n        g["weight_decay"] = a.wd if g["weight_decay"] else 0.0\n    if scaler.is_enabled() and st["scaler"]:\n        scaler.load_state_dict(st["scaler"])   # a bf16 (H100) checkpoint has an empty scaler state; start fresh\n    loader.load_state_dict(st["loader"])\n    step = st["step"]\n    run["metrics"], run["samples"] = st["metrics"], st["samples"]\n    rng_file = os.path.join(src, f"rng_{rank}.pt")\n    same_layout = st["world"] == world and st["micro"] == a.micro and st["dtype"] == str(dtype)\n    if os.path.exists(rng_file) and same_layout:\n        r = torch.load(rng_file, map_location="cpu", weights_only=False)   # cpu: a CUDA map_location breaks set_rng_state\n        random.setstate(r["python"])\n        np.random.set_state(r["numpy"])\n        torch.set_rng_state(r["torch"])\n        torch.cuda.set_rng_state(r["cuda"], dev)\n    if a.reset_loader_shard >= 0:\n        loader.load_state_dict({"shard": a.reset_loader_shard, "pos": loader.starts[a.reset_loader_shard], "epoch": 0})\n    prior_sec = st["metrics"][-1]["wallSec"] if st["metrics"] else 0.0\n    run["resumes"].append({"from": src, "step": step, "sameLayout": same_layout, "at": time.strftime("%Y-%m-%dT%H:%M:%S")})\n    if master:\n        print(f"resumed from {src} at step {step} (same layout: {same_layout})", flush=True)\n\n# ---- train -------------------------------------------------------------------------------------------------------\nfpt = flops_per_token(cfg, N)\nt_start = time.time()\nif master:\n    print(f"{a.size} params {N:,} | {torch.cuda.get_device_name(rank)} x{world} | {dtype} | micro {a.micro} x accum {accum}"\n          f" | {a.tokens_per_step:,} tokens/step | steps {step} -> {a.max_steps}", flush=True)\nwhile step < a.max_steps:\n    torch.cuda.synchronize()\n    t0 = time.time()\n    lr = lr_at(step)\n    for g in opt.param_groups:\n        g["lr"] = lr\n    tot = torch.zeros((), device=dev)\n    for i in range(accum):\n        x, y = loader.next()\n        ctx = contextlib.nullcontext() if i == accum - 1 else model.no_sync()\n        with ctx:\n            with torch.autocast("cuda", dtype=dtype):\n                loss = model(x, y) / accum\n            scaler.scale(loss).backward()\n        tot += loss.detach()\n    scaler.unscale_(opt)\n    gn = torch.nn.utils.clip_grad_norm_(raw.parameters(), a.clip)\n    scaler.step(opt)\n    scaler.update()\n    opt.zero_grad(set_to_none=True)\n    dist.all_reduce(tot, op=dist.ReduceOp.AVG)\n    step += 1\n    torch.cuda.synchronize()\n    dt = time.time() - t0\n    m = {"step": step, "tokens": step * a.tokens_per_step, "trainLoss": round(tot.item(), 5), "lr": lr,\n         "gradNorm": round(gn.item(), 4), "tokPerSec": round(a.tokens_per_step / dt), "stepSec": round(dt, 3),\n         "mfu": round(a.tokens_per_step / dt * fpt / (world * (989e12 if cap >= (9, 0) else 312e12 if cap >= (8, 0) else 65e12)), 4),\n         "memGB": round(torch.cuda.max_memory_allocated(dev) / 1e9, 2), "loaderShard": loader.shard,\n         "epoch": loader.epoch, "wallSec": round(prior_sec + time.time() - t_start, 1)}\n    if scaler.is_enabled():\n        m["lossScale"] = scaler.get_scale()\n    if step % a.val_every == 0 or step == a.max_steps:\n        m["valLoss"], m["valTokens"] = val_loss()\n        m["valLoss"] = round(m["valLoss"], 5)\n    if master and (step % a.sample_every == 0 or step == a.max_steps):\n        run["samples"] += samples(step)\n    run["metrics"].append(m)\n    if master and (step % 10 == 0 or "valLoss" in m):\n        print(json.dumps(m), flush=True)\n    if step % a.export_every == 0 or step in milestones or step == a.max_steps:\n        export_weights(step)\n    out_of_time = a.stop_after_sec and time.time() - t_start > a.stop_after_sec\n    stop = torch.tensor(int(bool(out_of_time)), device=dev)\n    dist.all_reduce(stop, op=dist.ReduceOp.MAX)          # every rank must agree to stop\n    if step % a.save_every == 0 or step in milestones or step == a.max_steps or stop.item():\n        save_ckpt(step, run)\n        write_run(run, step, prior_sec + time.time() - t_start)\n    if stop.item():\n        if master:\n            print(f"stopping at step {step} after {time.time() - t_start:.0f} s; rerun with --resume auto", flush=True)\n        break\n\nwrite_run(run, step, prior_sec + time.time() - t_start)\ndist.destroy_process_group()\n')
print('wrote', 'train.py')


In [ ]:
with open('hf_wrapper.py', 'w') as f:
    f.write('# hf_wrapper.py - wrap the course GPT as a Hugging Face PreTrainedModel so lm-evaluation-harness can score it.\n# This is the reference solution for the capstone\'s "write the HF-compatible wrapper" step.\nimport torch\nfrom transformers import PretrainedConfig, PreTrainedModel\nfrom transformers.modeling_outputs import CausalLMOutput\nfrom gpt import GPT\n\n\nclass LLTConfig(PretrainedConfig):\n    model_type = "llt-gpt"\n\n    def __init__(self, V=50257, d=768, L=12, H=12, ctx=1024, **kwargs):\n        self.V, self.d, self.L, self.H, self.ctx = V, d, L, H, ctx\n        self.vocab_size = V\n        self.max_position_embeddings = ctx      # lm-eval reads this to know the context length\n        super().__init__(**kwargs)\n\n\nclass LLTForCausalLM(PreTrainedModel):\n    config_class = LLTConfig\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.gpt = GPT(config.V, config.d, config.L, config.H, config.ctx)\n        self.post_init()\n\n    def _init_weights(self, module):\n        pass                                     # weights come from our checkpoint, never from HF init\n\n    def tie_weights(self, *args, **kwargs):\n        pass                                     # the GPT already shares one tensor between embedding and head\n\n    def get_input_embeddings(self):\n        return self.gpt.emb\n\n    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):\n        # Right padding (what lm-eval uses for causal models) cannot change earlier positions, so the mask is unused.\n        logits = self.gpt(input_ids)\n        loss = None\n        if labels is not None:\n            loss = torch.nn.functional.cross_entropy(logits[:, :-1].float().flatten(0, 1), labels[:, 1:].flatten())\n        return CausalLMOutput(loss=loss, logits=logits)\n\n\ndef load(weights_path, device="cuda", dtype=torch.float16):\n    """Load a weights_step_XXXXXX.pt file written by train.py."""\n    blob = torch.load(weights_path, map_location="cpu", weights_only=False)\n    model = LLTForCausalLM(LLTConfig(**blob["config"]))\n    model.gpt.load_state_dict(blob["state_dict"], strict=True)\n    return model.to(device=device, dtype=dtype).eval(), blob\n')
print('wrote', 'hf_wrapper.py')


In [ ]:
with open('eval.py', 'w') as f:
    f.write('# eval.py - score our checkpoints and openai-community/gpt2 in lm-evaluation-harness 0.4.13, same tasks and dtype.\n#\n#   python eval.py --weights /workspace/run/weights/weights_step_019073.pt --baseline\n#   python eval.py --weights "/workspace/run/weights/weights_step_*.pt" --tasks hellaswag     # a HellaSwag curve\n#\n# Writes one Run Library eval file per model into --out, and prints a comparison table.\nimport argparse, glob, importlib.metadata, json, os, time\nimport numpy as np\nimport torch\nimport lm_eval\nfrom lm_eval.models.huggingface import HFLM\nfrom transformers import AutoTokenizer\nfrom hf_wrapper import load\n\np = argparse.ArgumentParser()\np.add_argument("--weights", default="")                 # a file or a glob\np.add_argument("--baseline", action="store_true")       # also score openai-community/gpt2\np.add_argument("--tasks", default="hellaswag,arc_easy,piqa,lambada_openai")\np.add_argument("--dtype", default="auto", choices=["auto", "bf16", "fp16"])\np.add_argument("--batch-size", type=int, default=64)\np.add_argument("--limit", type=int, default=0)          # 0 = full tasks; tests only\np.add_argument("--val", default="")                     # val.bin for the wrapper-equals-native check\np.add_argument("--out", default="evals")\na = p.parse_args()\n\nos.makedirs(a.out, exist_ok=True)\ncap = torch.cuda.get_device_capability(0)\ndtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(a.dtype, torch.bfloat16 if cap >= (8, 0) else torch.float16)\ntok = AutoTokenizer.from_pretrained("openai-community/gpt2")\ntasks = a.tasks.split(",")\n\n\ndef score(lm, name, extra):\n    t0 = time.time()\n    res = lm_eval.simple_evaluate(model=lm, tasks=tasks, num_fewshot=0, limit=a.limit or None,\n                                  batch_size=a.batch_size, log_samples=False)\n    ev = {t: {k: v for k, v in r.items() if k not in ("alias", "name")} for t, r in res["results"].items()}\n    rec = {"runId": "lltCapstoneGPT2-eval-" + name, "module": "lltCapstoneGPT2", "label": "eval",\n           "hardware": {"gpu": torch.cuda.get_device_name(0)}, "env": {"lm_eval": importlib.metadata.version("lm_eval"),\n           "torch": torch.__version__, "dtype": str(dtype)}, "config": {"tasks": tasks, "num_fewshot": 0,\n           "limit": a.limit or None, "batch_size": a.batch_size, **extra}, "evals": ev,\n           "wallClockSec": round(time.time() - t0, 1), "date": time.strftime("%Y-%m-%d")}\n    with open(os.path.join(a.out, rec["runId"] + ".json"), "w") as f:\n        json.dump(rec, f, indent=1)\n    return ev\n\n\ndef headline(ev):\n    keys = {"hellaswag": "acc_norm,none", "arc_easy": "acc,none", "piqa": "acc,none", "lambada_openai": "acc,none"}\n    return {t: round(ev[t][k], 4) for t, k in keys.items() if t in ev}\n\n\nrows = []\nfor path in sorted(glob.glob(a.weights)) if a.weights else []:\n    model, blob = load(path, dtype=dtype)\n    if a.val:                                   # the wrapper must give the same loss as the native model\n        w = torch.from_numpy(np.memmap(a.val, dtype=np.uint16, mode="r")[:4 * 1024 + 1].astype(np.int64)).cuda()\n        x, y = w[:-1].view(4, 1024), w[1:].view(4, 1024)\n        with torch.no_grad():\n            native = model.gpt(x, y).item()\n            last = torch.nn.functional.cross_entropy(model(input_ids=x).logits.float().flatten(0, 1), y.flatten()).item()\n        print(f"wrapper check: native {native:.6f}  wrapper logits {last:.6f}  diff {abs(native - last):.2e}", flush=True)\n        assert abs(native - last) < 1e-3, "the wrapper\'s logits differ from the native model"\n    lm = HFLM(pretrained=model, tokenizer=tok, backend="causal", batch_size=a.batch_size, max_length=1024)\n    name = "ours-step%06d" % blob["step"]\n    ev = score(lm, name, {"weights": os.path.basename(path), "step": blob["step"], "tokens": blob["tokens"]})\n    rows.append((name, blob["tokens"], headline(ev)))\n    del lm, model\n    torch.cuda.empty_cache()\n\nif a.baseline:\n    lm = HFLM(pretrained="openai-community/gpt2", dtype="bfloat16" if dtype == torch.bfloat16 else "float16",\n              batch_size=a.batch_size, device="cuda")\n    rows.append(("openai-community/gpt2", None, headline(score(lm, "openai-gpt2", {"model": "openai-community/gpt2"}))))\n\nfor name, tokens, h in rows:\n    print(f"{name:<24} tokens {tokens or \'\':>14}  " + "  ".join(f"{t} {v}" for t, v in h.items()), flush=True)\n')
print('wrote', 'eval.py')


In [ ]:

D = "/kaggle/working/data"
ARGS = ["--data", D, "--micro", "8", "--tokens-per-step", "65536", "--max-steps", "12", "--warmup", "2",
        "--stable-end", "8", "--save-every", "6", "--milestones", "6,12", "--export-every", "12",
        "--val-every", "6", "--val-tokens", "131072", "--sample-every", "12"]
TR = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2", "train.py"]
out = sh("prep", [sys.executable, "prep_fineweb.py", "--out", D, "--files", "2", "--max-docs", "1000",
                  "--val-tokens", "300000", "--workers", "2"], timeout=400)
RUN["data"] = result_line(out)
print(json.load(open(D + "/index.json"))["train"])


In [ ]:

sh("runA", TR + ARGS + ["--out", "/kaggle/working/A", "--run-id", "resumeA"], timeout=600)
ck = "/kaggle/working/A/ckpt/step_000006"
RUN["ckptFiles"] = {f: os.path.getsize(os.path.join(ck, f)) for f in sorted(os.listdir(ck))}
RUN["ckptGB"] = round(sum(RUN["ckptFiles"].values()) / 1e9, 3)
print(RUN["ckptFiles"], RUN["ckptGB"])
save()


In [ ]:

sh("runB", TR + ARGS + ["--out", "/kaggle/working/B", "--run-id", "resumeB", "--resume", "/kaggle/working/A/ckpt/step_000006"],
   timeout=600)
A = json.load(open("/kaggle/working/A/resumeA.json"))["metrics"]
B = json.load(open("/kaggle/working/B/resumeB.json"))["metrics"]
cmp = []
for ma, mb in zip(A[6:], [m for m in B if m["step"] > 6]):
    row = {"step": ma["step"], "shardA": ma["loaderShard"], "shardB": mb["loaderShard"]}
    for k in ("trainLoss", "gradNorm", "valLoss", "lossScale", "lr"):
        if k in ma:
            row[k] = [ma[k], mb.get(k)]
    cmp.append(row)
RUN["resumeCompare"] = cmp
diffs = [abs(r["trainLoss"][0] - r["trainLoss"][1]) for r in cmp]
RUN["resumeMaxAbsDiffTrainLoss"] = max(diffs)
RUN["resumeBitwiseTrainLoss"] = all(d == 0 for d in diffs)
for r in cmp:
    print(r)
print("max abs diff", max(diffs), "bitwise", RUN["resumeBitwiseTrainLoss"])
save()


In [ ]:

if time.time() - T_START < 9 * 60:
    sh("runC_branch", TR + ARGS[:6] + ["--max-steps", "9", "--warmup", "2", "--stable-end", "6", "--save-every", "9",
       "--milestones", "9", "--export-every", "9", "--val-every", "3", "--val-tokens", "131072", "--sample-every", "9",
       "--out", "/kaggle/working/C", "--run-id", "branchC", "--resume", "/kaggle/working/A/ckpt/step_000006"], timeout=600)
    C = json.load(open("/kaggle/working/C/branchC.json"))["metrics"]
    RUN["branch"] = [{k: m.get(k) for k in ("step", "lr", "trainLoss", "valLoss")} for m in C if m["step"] > 6]
    print(RUN["branch"])
    save()


In [ ]:

if time.time() - T_START < 11 * 60:
    sh("pip_lmeval", [sys.executable, "-m", "pip", "install", "-q", "lm-eval==0.4.13"], timeout=300)
    env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    out = sh("eval_wrapper", [sys.executable, "eval.py", "--weights", "/kaggle/working/A/weights/weights_step_000012.pt",
                              "--val", D + "/val.bin", "--limit", "50", "--batch-size", "32",
                              "--out", "/kaggle/working/evals"], timeout=600, env=env1)
import shutil
for d in ("/kaggle/working/data", "/kaggle/working/A/ckpt", "/kaggle/working/B/ckpt", "/kaggle/working/C/ckpt",
          "/kaggle/working/A/weights", "/kaggle/working/B/weights", "/kaggle/working/C/weights"):
    shutil.rmtree(d, ignore_errors=True)
print("total min", round((time.time() - T_START) / 60, 1))
save()
